# 07 — Análisis de separabilidad con LOSO (genérico)

**Pregunta:** ¿qué tan separables son los agarres y cuáles se confunden entre sí?

La validación es **LOSO** (Leave-One-Subject-Out) con el sujeto definido como
`subject + origin_db` (cada sujeto se deja fuera una vez; sus filas nunca están en train y test a
la vez, así no hay fuga de información) y el notebook funciona con **cualquier dataset** del proyecto:

| `DATASET` | Archivo | Variables | Estilo de nombre | Unidad |
|---|---|---|---|---|
| `channels` | `data/prueba/intact_yj_mean_wnd.csv` | 48 | `Channel 1_VAR_STD` | ventana |
| `pca` | `data/prueba/df_pca_wnd_mean.csv` | 6 | `VAR_STD` | ventana |
| `flat` | `data/prueba/df_flat_wnd_mean.csv` | 150 | `VAR_STD_w0` | fila (sujeto × agarre) |
| `stats` | `data/prueba/df_stats_wnd_mean.csv` | 42 | `VAR_STD_median` | fila (sujeto × agarre) |
| `stats_max` | `data/prueba/df_stats_wnd_max.csv` | 42 | `VAR_STD_max` | fila (sujeto × agarre) |

**Uso:** cambiar solo `DATASET` en la celda de configuración. Las columnas meta (`window` es
opcional), los nombres de features, la paleta de colores, las etiquetas y la carpeta de figuras
(`figures/separabilidad_<archivo>/`) se **auto-detectan**.

**Todo está implementado como funciones** (celdas de funciones) reutilizables desde otros
notebooks o scripts; las secciones de análisis solo las invocan.

- **Clases**: 7 agarres, sin reposo (`stimulus = 0`).
- **Validación**: LOSO — un fold por sujeto (87 folds), grupos `subject + origin_db`
  (los IDs de sujeto se repiten entre bases NinaPro).
- **Figuras**: se guardan en `figures/separabilidad_<archivo>/` (no se muestran en el notebook).

| Sección | Pregunta |
|---|---|
| 1. Carga | ¿Qué contiene el dataset? (auto-detectado) |
| 2. Fisher Discriminant Ratio | ¿Qué variables discriminan más? |
| 3. LDA multiclase con LOSO | ¿Se separan los agarres en sujetos no vistos? |
| 4. Solapamiento por pares | ¿Qué pares de agarres se confunden? |
| 5. Silhouette por clase | ¿Cada agarre forma un grupo compacto? |
| 6. Proyección LDA | Visualización descriptiva |
| 7. Resumen | Cifras principales |

In [1]:
from _provenance import Sello
sello = Sello('07_separability_analysis.ipynb')

In [2]:
import os
import re
import itertools
import warnings
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (silhouette_score, silhouette_samples, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ======================= Dataset a analizar =======================
# Cambia SOLO esta linea: 'channels' | 'pca' | 'flat' | 'stats' | 'stats_max'
DATASET = 'channels'

DATASETS = {
    'channels':  '../data/prueba/intact_yj_mean_wnd.csv',  # 8 canales sin fusionar (48 vars)
    'pca':       '../data/prueba/df_pca_wnd_mean.csv',     # PCA por feature y ventana (6 vars)
    'flat':      '../data/prueba/df_flat_wnd_mean.csv',    # ventanas aplanadas (150 vars)
    'stats':     '../data/prueba/df_stats_wnd_mean.csv',   # estadisticos por ventana (42 vars)
    'stats_max': '../data/prueba/df_stats_wnd_max.csv',
}
if DATASET not in DATASETS:
    raise KeyError(f'DATASET={DATASET!r} no esta en DATASETS: {sorted(DATASETS)}')
DATA_PATH = DATASETS[DATASET]

META_COLS = ['origin_db', 'subject', 'stimulus', 'window']   # 'window' es opcional
REST = 0

# ================= Validacion: LOSO ==================
# Leave-One-Subject-Out: un fold por sujeto (subject + origin_db); cada sujeto
# se evalua exactamente una vez con un modelo entrenado con todos los demas.

# Shrinkage de Ledoit-Wolf para LDA ('auto' recomendado | None = LDA clasica).
SHRINKAGE = 'auto'
# --------------------------------------------------------------

# Paleta tab10: colores bien distinguibles entre si (azul, naranja, verde,
# rojo, morado, marron, rosa, ...) — la anterior mezclaba rojo/naranja y azul/cian.
STIM_COLORS = ['#1F77B4', '#FF7F0E', '#2CA02C', '#D62728',
               '#9467BD', '#8C564B', '#E377C2', '#BCBD22', '#17BECF', '#7F7F7F']

print(f'DATASET   = {DATASET!r}')
print(f'DATA_PATH = {DATA_PATH}')
print('Validacion = LOSO (Leave-One-Subject-Out)')
print(f'SHRINKAGE = {SHRINKAGE!r}')

# Fecha de generacion de las figuras (sello en la esquina de cada PDF)
GEN_DATE = datetime.now().strftime('%Y-%m-%d %H:%M')

DATASET   = 'channels'
DATA_PATH = ../data/prueba/intact_yj_mean_wnd.csv
Validacion = LOSO (Leave-One-Subject-Out)
SHRINKAGE = 'auto'


## Funciones — datos

- `detect_feature_style()` identifica el patrón de nombres de columnas
  (`Channel 1_VAR_STD` / `VAR_STD_w0` / `VAR_STD_median` / `VAR_STD`) y devuelve la función
  `feat_of()` que extrae el nombre base de cada feature.
- `load_dataset()` lee el CSV, detecta las columnas meta presentes (`window` es opcional), quita el
  reposo, crea el grupo `subject + origin_db` y devuelve el DataFrame más un diccionario `info` con
  el esquema detectado (unidad, estilo, tipos de feature, carpeta de figuras).

In [3]:
_CHANNEL_RE = re.compile(r'^Channel \d+_')
_FLAT_RE = re.compile(r'_w\d+$')
_STAT_RE = re.compile(r'_(median|min|max|first|last|slope|std)$')


def detect_feature_style(feature_cols):
    """Detecta el patron de nombres de las columnas de features.

    Devuelve (estilo, feat_of) donde feat_of extrae el nombre base de la feature:
        'Channel 1_VAR_STD' -> ('channel', ...) -> 'VAR_STD'
        'VAR_STD_w0'        -> ('flat',    ...) -> 'VAR_STD'
        'VAR_STD_median'    -> ('stats',   ...) -> 'VAR_STD'
        'VAR_STD'           -> ('plain',   ...) -> 'VAR_STD'
    """
    if feature_cols and all(_CHANNEL_RE.match(c) for c in feature_cols):
        return 'channel', lambda c: c.split('_', 1)[1]
    if feature_cols and all(_FLAT_RE.search(c) for c in feature_cols):
        return 'flat', lambda c: c.rsplit('_w', 1)[0]
    if feature_cols and all(_STAT_RE.search(c) for c in feature_cols):
        return 'stats', lambda c: c.rsplit('_', 1)[0]
    return 'plain', lambda c: c


def load_dataset(data_path, rest=REST):
    """Lee el CSV, detecta meta/features, quita el reposo y crea el grupo de validacion.

    Devuelve (df, info); info resume el esquema detectado: columnas, estilo de nombres,
    unidad de muestra ('ventanas' o 'filas'), tipos de feature y carpeta de figuras.
    """
    df = pd.read_csv(data_path)
    shape_raw = df.shape

    missing = [c for c in ('stimulus', 'subject', 'origin_db') if c not in df.columns]
    if missing:
        raise KeyError(f'Faltan columnas obligatorias en {data_path}: {missing}')

    meta_cols = [c for c in META_COLS if c in df.columns]
    feature_cols = [c for c in df.columns if c not in meta_cols]
    if not feature_cols:
        raise KeyError(f'No se detectaron columnas de features en {data_path}')

    df = df[df['stimulus'] != rest].copy().reset_index(drop=True)
    shape = df.shape
    df['group'] = df['subject'].astype(str) + '_' + df['origin_db'].astype(str)

    style, feat_of = detect_feature_style(feature_cols)
    info = {
        'data_path': data_path,
        'tag': Path(data_path).stem,
        'save_dir': f'../figures/separabilidad_{Path(data_path).stem}',
        'shape_raw': shape_raw,
        'shape': shape,
        'meta_cols': meta_cols,
        'feature_cols': feature_cols,
        'n_features': len(feature_cols),
        'has_window': 'window' in df.columns,
        'unit': 'ventanas' if 'window' in df.columns else 'filas',
        'style': style,
        'feat_of': feat_of,
        'feature_types': sorted({feat_of(c) for c in feature_cols}),
        'channels': sorted({c.split('_', 1)[0] for c in feature_cols}) if style == 'channel' else [],
    }
    return df, info

## Funciones — métricas

- `fisher_ratio()` / `fdr_global()` y `compute_fdr()`: ratio de Fisher por variable y agregado.
- `compute_silhouette()`: silhouette por muestra, global y por clase.

El FDR global y el silhouette dependen de la escala de cada variable, así que se calculan sobre
las variables **estandarizadas** (z-score); el FDR por variable no cambia con la escala.

Se calculan sobre `X` y `y` sin importar cuántas variables tenga el dataset.

In [4]:
def fisher_ratio(X_col, y):
    # FDR univariado: SS_between / SS_within
    classes = np.unique(y)
    overall_mean = X_col.mean()
    ss_between = sum((y == c).sum() * (X_col[y == c].mean() - overall_mean) ** 2 for c in classes)
    ss_within = sum(((X_col[y == c] - X_col[y == c].mean()) ** 2).sum() for c in classes)
    return ss_between / ss_within if ss_within > 0 else np.nan


def fdr_global(X, y):
    # FDR agregado sobre todas las dimensiones
    classes = np.unique(y)
    overall_mean = X.mean(axis=0)
    ss_between = np.zeros(X.shape[1])
    ss_within = np.zeros(X.shape[1])
    for c in classes:
        X_c = X[y == c]
        ss_between += len(X_c) * (X_c.mean(axis=0) - overall_mean) ** 2
        ss_within += ((X_c - X_c.mean(axis=0)) ** 2).sum(axis=0)
    return ss_between.sum() / ss_within.sum() if ss_within.sum() > 0 else np.nan


def zscore(X):
    """Estandariza cada columna (media 0, desviacion 1); columnas constantes quedan en 0."""
    sd = X.std(axis=0)
    return (X - X.mean(axis=0)) / np.where(sd > 0, sd, 1)


def compute_fdr(X, y, feature_cols):
    """FDR por variable + FDR global. Devuelve (fdr_df ordenado, fdr_global_value).

    El FDR global se calcula con X estandarizado para que todas las variables pesen igual.
    """
    fdr_values = np.array([fisher_ratio(X[:, i], y) for i in range(X.shape[1])])
    fdr_global_value = fdr_global(zscore(X), y)
    fdr_df = (pd.DataFrame({'Variable': feature_cols, 'FDR': fdr_values})
              .sort_values('FDR', ascending=False).reset_index(drop=True))
    fdr_df.index = fdr_df.index + 1
    return fdr_df, fdr_global_value


def compute_silhouette(X, y):
    """Silhouette por muestra, global y agregada por clase (sobre X estandarizado).

    Devuelve (sil_samples, sil_global, sil_class).
    """
    sil_samples = silhouette_samples(zscore(X), y)
    sil_global = float(np.mean(sil_samples))
    sil_class = (pd.DataFrame({'stimulus': y, 'silhouette': sil_samples})
                 .groupby('stimulus')['silhouette']
                 .agg(media='mean', desv='std', n='count').round(4))
    sil_class.index.name = 'Estímulo'
    return sil_samples, sil_global, sil_class

## Funciones — LDA y validación

- `make_lda()`: constructor de la LDA (`'lsqr'` para clasificar, `'eigen'` para proyectar).
- `run_lda_loso()`: LDA multiclase con LOSO (un fold por sujeto); devuelve accuracy por sujeto, predicciones acumuladas de prueba y la matriz de confusión normalizada.
- `lda_pair_error()` / `pairwise_overlap()`: error de una LDA binaria por cada par de agarres.
- `lda_projection()`: proyección LDA descriptiva (todas las etiquetas) con el poder discriminante
  de cada componente.

La validación LOSO no tiene parámetros aleatorios (es determinista) y funciona con cualquier número de
muestras o variables.

In [5]:
def make_lda(shrinkage=SHRINKAGE, for_transform=False):
    # 'lsqr' para clasificar; 'eigen' cuando se necesita .transform() (proyeccion)
    solver = 'eigen' if for_transform else 'lsqr'
    if shrinkage is None and not for_transform:
        solver = 'svd'
    return LinearDiscriminantAnalysis(solver=solver, shrinkage=shrinkage)


def run_lda_loso(X, y, groups, shrinkage=SHRINKAGE):
    """LDA multiclase con Leave-One-Subject-Out.

    En cada fold se entrena con todos los sujetos menos uno y se predicen las muestras del
    sujeto excluido; cada sujeto se evalua exactamente una vez.

    Devuelve un dict con predicciones acumuladas, accuracy global/balanced, accuracy por
    sujeto y la matriz de confusión normalizada por fila.
    """
    rows = []
    y_true_parts, y_pred_parts = [], []

    for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups):
        clf = make_lda(shrinkage)
        clf.fit(X[train_idx], y[train_idx])
        pred = clf.predict(X[test_idx])
        y_true_parts.append(y[test_idx])
        y_pred_parts.append(pred)
        rows.append({'grupo': groups[test_idx][0],
                     'accuracy': accuracy_score(y[test_idx], pred),
                     'n_muestras': len(test_idx)})

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    stimuli = sorted(np.unique(y).tolist())
    cm = confusion_matrix(y_true_all, y_pred_all, labels=stimuli, normalize='true')

    return {
        'y_true_all': y_true_all,
        'y_pred_all': y_pred_all,
        'per_group_df': pd.DataFrame(rows).sort_values('grupo').reset_index(drop=True),
        'cm': cm,
        'stimuli': stimuli,
        'acc_global': accuracy_score(y_true_all, y_pred_all),
        'bacc_global': balanced_accuracy_score(y_true_all, y_pred_all),
        'n_groups': len(np.unique(groups)),
    }


def lda_pair_error(i, j, X, y, groups, shrinkage=SHRINKAGE):
    # Tasa de error de una LDA entrenada solo con los agarres i y j (LOSO)
    mask = np.isin(y, [i, j])
    Xp, yp, gp = X[mask], y[mask], groups[mask]
    errors, total = 0, 0
    for train_idx, test_idx in LeaveOneGroupOut().split(Xp, yp, gp):
        clf = make_lda(shrinkage)
        clf.fit(Xp[train_idx], yp[train_idx])
        errors += int((clf.predict(Xp[test_idx]) != yp[test_idx]).sum())
        total += len(test_idx)
    return errors / total if total > 0 else np.nan


def pairwise_overlap(X, y, stimuli, groups, **kwargs):
    """Matriz de error binario por pares de agarres (simetrica, diagonal 0)."""
    n = len(stimuli)
    overlap = np.zeros((n, n))
    for a, b in itertools.combinations(range(n), 2):
        overlap[a, b] = overlap[b, a] = lda_pair_error(stimuli[a], stimuli[b],
                                                       X, y, groups, **kwargs)
    return overlap


def fisher_axis(z, y):
    """Razon entre-clases / intra-clases de una proyeccion 1D (siempre >= 0)."""
    mu = z.mean()
    sb = sw = 0.0
    for c in np.unique(y):
        zc = z[y == c]
        sb += len(zc) * (zc.mean() - mu) ** 2
        sw += ((zc - zc.mean()) ** 2).sum()
    return sb / sw


def lda_projection(X, y, shrinkage=SHRINKAGE):
    """Proyeccion LDA descriptiva (usa todas las etiquetas: es optimista).

    Devuelve (X_lda, evr, var_df) con evr = proporcion de poder discriminante por componente.
    """
    lda = make_lda(shrinkage, for_transform=True)
    X_lda = lda.fit_transform(X, y)
    pod = np.array([fisher_axis(X_lda[:, k], y) for k in range(X_lda.shape[1])])
    evr = pod / pod.sum()
    var_df = pd.DataFrame({
        'Componente': [f'LD{i + 1}' for i in range(len(evr))],
        'Poder discriminante (λ)': pod,
        'Proporción': evr,
        'Acumulada': np.cumsum(evr),
    })
    return X_lda, evr, var_df

## Funciones — visualización

Todas guardan el PDF en la carpeta del dataset y **no muestran la figura en el notebook**
(`savefig` + `close`, sin `plt.show()`):

Cada PDF lleva en la esquina inferior derecha la fecha de generación (`Generado: YYYY-MM-DD HH:MM`).

| Función | PDF |
|---|---|
| `plot_fdr()` | `fdr_features.pdf` |
| `plot_confusion()` | `lda_loso_confusion.pdf` |
| `plot_overlap()` | `overlap_matrix.pdf` |
| `plot_silhouette()` | `silhouette_per_class.pdf` |
| `plot_lda_projection()` | `lda_projection.pdf` |
| `plot_lda_projection_3d()` | `lda_projection_3d.pdf` |

La proyección LDA 2D (`lda_projection.pdf`) dibuja los **centroides** de cada agarre (marcador `X` con borde negro); la 3D (`lda_projection_3d.pdf`, ejes LD1–LD3) solo los puntos. La paleta por defecto (`STIM_COLORS`) usa colores tab10 bien distinguibles.

`print_summary()` imprime las cifras principales de todas las secciones.

In [6]:
def _save(fig, save_dir, name):
    """Guarda la figura en PDF dentro de save_dir y libera la memoria (sin plt.show).

    Anade en la esquina inferior derecha la fecha de generacion (GEN_DATE).
    """
    fig.text(0.99, 0.01, f'Generado: {GEN_DATE}', ha='right', va='bottom',
             fontsize=7, color='#666666', transform=fig.transFigure)
    os.makedirs(save_dir, exist_ok=True)
    fig.savefig(f'{save_dir}/{name}.pdf', dpi=300, bbox_inches='tight')
    plt.close(fig)


def plot_fdr(fdr_df, fdr_global_value, feat_of, feature_types, save_dir):
    """Barras de FDR por variable; colores por tipo de feature (si hay <=15 tipos)."""
    n_types = len(feature_types)
    use_legend = n_types <= 15
    if use_legend:
        palette = dict(zip(feature_types, sns.color_palette('tab10', n_types)))
        colors = [palette[feat_of(v)] for v in fdr_df['Variable']]
    else:
        colors = '#1E88E5'

    fig, ax = plt.subplots(figsize=(13, 4.5), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    ax.set_facecolor('#FFFFFF')

    ax.bar(fdr_df['Variable'], fdr_df['FDR'], color=colors, edgecolor='white', width=0.75)
    ax.axhline(fdr_global_value, color='tomato', linestyle='--', linewidth=1.2)

    if use_legend:
        handles = [plt.Rectangle((0, 0), 1, 1, fc=palette[t], ec='white')
                   for t in feature_types]
        handles.append(plt.Line2D([0], [0], color='tomato', ls='--', lw=1.2))
        labels = feature_types + [f'FDR global ({fdr_global_value:.3f})']
        ax.legend(handles, labels, frameon=True, fontsize=7, ncol=2, loc='upper right')
    else:
        ax.legend([plt.Line2D([0], [0], color='tomato', ls='--', lw=1.2)],
                  [f'FDR global ({fdr_global_value:.3f})'], frameon=True, fontsize=8)

    ax.set_ylabel('FDR (entre / dentro de clases)', fontweight='bold')
    ax.set_title('Fisher Discriminant Ratio por variable', fontweight='bold')
    ax.tick_params(axis='x', rotation=90)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#CCCCCC')
    ax.tick_params(colors='#444444', labelsize=6)
    plt.tight_layout()
    _save(fig, save_dir, 'fdr_features')


def plot_confusion(cm_df, per_group_df, chance, save_dir):
    """Heatmap de la matriz de confusion + histogramo de accuracy por sujeto."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300,
                             gridspec_kw={'width_ratios': [1.25, 1]})
    fig.patch.set_facecolor('#F7F9FC')

    ax = axes[0]
    sns.heatmap(cm_df, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1,
                annot_kws={'fontsize': 8},
                linewidths=0.5, linecolor='#FFFFFF',
                cbar_kws={'label': 'Proporción (por fila)'}, ax=ax)
    ax.set_title('Matriz de confusión LDA — LOSO (normalizada por fila)',
                 fontweight='bold')
    ax.set_xlabel('Predicho', fontweight='bold')
    ax.set_ylabel('Real', fontweight='bold')

    ax = axes[1]
    ax.set_facecolor('#FFFFFF')
    ax.hist(per_group_df['accuracy'], bins=np.linspace(0, 1, 9), color='#1E88E5',
            edgecolor='white', alpha=0.9)
    ax.axvline(chance, color='tomato', linestyle='--', linewidth=1.2,
               label=f'Azar ({chance:.3f})')
    ax.axvline(per_group_df['accuracy'].mean(), color='#333333', linestyle='-',
               linewidth=1.2, label=f'Media ({per_group_df["accuracy"].mean():.3f})')
    ax.set_xlabel('Accuracy del sujeto de prueba', fontweight='bold')
    ax.set_ylabel('N.º de sujetos', fontweight='bold')
    ax.set_title('Distribución de accuracy por sujeto', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=True, fontsize=8)

    plt.tight_layout()
    _save(fig, save_dir, 'lda_loso_confusion')


def plot_overlap(overlap_df, save_dir):
    """Heatmap de la matriz de error binario por pares."""
    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    ax.set_facecolor('#FFFFFF')

    sns.heatmap(overlap_df, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0,
                linewidths=0.5, linecolor='#FFFFFF',
                cbar_kws={'label': 'Tasa de error'}, ax=ax)
    ax.set_title('Matriz de solapamiento entre agarres — Error LDA por pares (LOSO)',
                 fontweight='bold')
    ax.set_xlabel('Estímulo', fontweight='bold')
    ax.set_ylabel('Estímulo', fontweight='bold')
    ax.tick_params(colors='#444444', labelsize=9)
    plt.tight_layout()
    _save(fig, save_dir, 'overlap_matrix')


def plot_silhouette(sil_class, stimuli, stim_color_map, sil_global, save_dir):
    """Barras de silhouette medio por clase con desviacion estandar y silhouette global."""
    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    ax.set_facecolor('#FFFFFF')

    vals = sil_class.loc[stimuli, 'media']
    errs = sil_class.loc[stimuli, 'desv']
    bars = ax.bar([str(s) for s in stimuli], vals,
                  color=[stim_color_map[s] for s in stimuli], edgecolor='white', width=0.7,
                  yerr=errs, capsize=4, error_kw=dict(ecolor='#555555', lw=0.8))
    ax.axhline(0, color='#333333', linewidth=0.8)
    ax.axhline(sil_global, color='tomato', linestyle='--', linewidth=1.2,
               label=f'Silhouette global = {sil_global:.3f}')

    offset = 0.01
    for bar, val in zip(bars, vals):
        va = 'bottom' if val >= 0 else 'top'
        ax.text(bar.get_x() + bar.get_width() / 2,
                val + (offset if val >= 0 else -offset),
                f'{val:.3f}', ha='center', va=va, fontsize=8, fontweight='bold')

    lim = max(np.abs(vals - errs).max(), np.abs(vals + errs).max()) * 1.15
    ax.set_ylim(-lim, lim)

    ax.set_xlabel('Estímulo (agarre)', fontweight='bold')
    ax.set_ylabel('Silhouette medio', fontweight='bold')
    ax.set_title('Silhouette por clase (sin reposo)', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#CCCCCC')
    ax.tick_params(colors='#444444', labelsize=9)
    ax.legend(frameon=True, fontsize=8)
    plt.tight_layout()
    _save(fig, save_dir, 'silhouette_per_class')


def lda_centroids(X_lda, y, stimuli, n_dims):
    """Centroide por agarre en las primeras n_dims componentes LDA."""
    return {stim: X_lda[y == stim, :n_dims].mean(axis=0) for stim in stimuli}


def plot_lda_projection(X_lda, y, stimuli, evr, stim_colors, save_dir):
    """Scatter LD1 x LD2 coloreado por agarre + centroides por clase."""
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    ax.set_facecolor('#FFFFFF')

    for stim, color in zip(stimuli, stim_colors):
        mask = y == stim
        ax.scatter(X_lda[mask, 0], X_lda[mask, 1], s=14, alpha=0.75,
                   edgecolor='white', linewidth=0.3, color=color,
                   label=f'Estímulo {stim}')

    # Centroides por agarre (marcador X con borde negro)
    cents = lda_centroids(X_lda, y, stimuli, 2)
    for stim, color in zip(stimuli, stim_colors):
        cx, cy = cents[stim]
        ax.scatter(cx, cy, marker='X', s=160, color=color,
                   edgecolor='black', linewidth=1.2, zorder=5)
    ax.scatter([], [], marker='X', s=160, color='#888888',
               edgecolor='black', linewidth=1.2, label='Centroide')

    ax.set_xlabel(f'LD1 ({evr[0] * 100:.1f}%)', fontweight='bold')
    ax.set_ylabel(f'LD2 ({evr[1] * 100:.1f}%)', fontweight='bold')
    ax.set_title('Proyección LDA (descriptiva) — sin reposo', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#CCCCCC')
    ax.tick_params(colors='#444444', labelsize=8)
    ax.legend(title='Estímulo', frameon=True, loc='best', fontsize=8, title_fontsize=9)
    plt.tight_layout()
    _save(fig, save_dir, 'lda_projection')


def plot_lda_projection_3d(X_lda, y, stimuli, evr, stim_colors, save_dir):
    """Scatter LD1 x LD2 x LD3 coloreado por agarre (sin centroides)."""
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registra projection='3d')

    fig = plt.figure(figsize=(11, 7), dpi=300)
    fig.patch.set_facecolor('#F7F9FC')
    ax = fig.add_subplot(111, projection='3d')
    ax.set_facecolor('#FFFFFF')

    for stim, color in zip(stimuli, stim_colors):
        mask = y == stim
        ax.scatter(X_lda[mask, 0], X_lda[mask, 1], X_lda[mask, 2],
                   s=14, alpha=0.75, edgecolor='white', linewidth=0.3,
                   color=color, depthshade=False, label=f'Estímulo {stim}')

    ax.set_xlabel(f'LD1 ({evr[0] * 100:.1f}%)', fontweight='bold')
    ax.set_ylabel(f'LD2 ({evr[1] * 100:.1f}%)', fontweight='bold')
    ax.set_zlabel(f'LD3 ({evr[2] * 100:.1f}%)', fontweight='bold')
    ax.set_title(f'Proyección LDA 3D (descriptiva) — sin reposo\n'
                 f'LD1+LD2+LD3 = {evr[:3].sum() * 100:.1f}% del poder discriminante',
                 fontweight='bold')
    ax.view_init(elev=30, azim=-60)
    ax.legend(title='Estímulo', frameon=True, loc='upper left',
              fontsize=8, title_fontsize=9)
    plt.tight_layout()
    _save(fig, save_dir, 'lda_projection_3d')


def print_summary(info, fdr_df, fdr_global_value, lda_res, overlap,
                  sil_global, sil_class, chance):
    """Imprime las cifras principales de todas las secciones."""
    stimuli = lda_res['stimuli']
    per_group_df = lda_res['per_group_df']
    cm = lda_res['cm']
    n = len(stimuli)

    fdr_var = fdr_df.set_index('Variable')['FDR']
    sil_sorted = sil_class['media'].sort_values(ascending=False)

    conf_pairs = sorted(
        [{'a': stimuli[x], 'b': stimuli[y], 'v': (cm[x, y] + cm[y, x]) / 2}
         for x, y in itertools.combinations(range(n), 2)],
        key=lambda d: d['v'], reverse=True)
    pair_sorted = sorted(
        [{'a': stimuli[x], 'b': stimuli[y], 'v': overlap[x, y]}
         for x, y in itertools.combinations(range(n), 2)],
        key=lambda d: d['v'], reverse=True)

    print(f"Configuración: {info['n_features']} variables ({info['style']}), "
          f"SHRINKAGE={SHRINKAGE!r}, LOSO, "
          f"{lda_res['n_groups']} sujetos, {n} agarres, "
          f"{info['shape'][0]} {info['unit']}\n")
    print('Fisher Discriminant Ratio')
    print(f'  FDR global = {fdr_global_value:.3f}')
    print('  Variables más discriminativas: '
          + ', '.join(f'{v} ({x:.3f})' for v, x in fdr_var.head(3).items()))
    print('  Variables menos discriminativas: '
          + ', '.join(f'{v} ({x:.3f})' for v, x in fdr_var.tail(2).items()) + '\n')
    print('LDA multiclase LOSO')
    print(f'  Accuracy global = {lda_res["acc_global"]:.3f} | por sujeto = '
          f'{per_group_df["accuracy"].mean():.3f} ± {per_group_df["accuracy"].std():.3f} '
          f'(azar = {chance:.3f}); balanced accuracy = {lda_res["bacc_global"]:.3f}')
    recall = np.diag(cm)
    print(f'  Mejor reconocido: {stimuli[int(np.argmax(recall))]} ({recall.max():.3f}) | '
          f'peor: {stimuli[int(np.argmin(recall))]} ({recall.min():.3f})')
    print('  Pares más confundidos: '
          + ', '.join(f'{d["a"]}–{d["b"]} ({d["v"]:.3f})' for d in conf_pairs[:3]) + '\n')
    print('Solapamiento por pares (LDA binaria LOSO)')
    print(f'  Error medio = {overlap[np.triu_indices(n, 1)].mean():.3f}')
    print('  Más solapados: '
          + ', '.join(f'{d["a"]}–{d["b"]} ({d["v"]:.3f})' for d in pair_sorted[:3]))
    print('  Menos solapados: '
          + ', '.join(f'{d["a"]}–{d["b"]} ({d["v"]:.3f})' for d in pair_sorted[-3:]) + '\n')
    print('Silhouette')
    print(f'  Global = {sil_global:.3f}; mejor clase {sil_sorted.index[0]} '
          f'({sil_sorted.iloc[0]:.3f}), peor {sil_sorted.index[-1]} ({sil_sorted.iloc[-1]:.3f})')

## 1. Carga

`load_dataset()` lee el CSV, auto-detecta meta y features, quita el reposo y crea el grupo
`subject + origin_db` (los IDs de sujeto se repiten entre bases NinaPro, por eso el grupo es la
combinación). La unidad de muestra (`ventanas` o `filas`) depende de si el dataset trae la columna
`window`.


In [7]:
df, info = load_dataset(DATA_PATH)

FEATURE_COLS = info['feature_cols']
N_FEATURES = info['n_features']
SAVE_DIR = info['save_dir']

print(f"Dataset:    {DATA_PATH} (DATASET = {DATASET!r})")
print(f"Estilo:     {info['style']} | {N_FEATURES} variables | unidad: {info['unit']}")
print(f"Meta:       {info['meta_cols']}")
print(f"Features:   {info['feature_types']}")
if info['channels']:
    print(f"Canales:    {info['channels']}")
print(f"Figuras en: {SAVE_DIR}")
print(f"Shape original: {info['shape_raw']} | sin reposo: {info['shape']}")
print(f"Clases: {sorted(df['stimulus'].unique().tolist())}")
display(df['stimulus'].value_counts().sort_index().to_frame('n_muestras'))

counts = df.groupby('group').size()
print(f'Grupos (subject_origin_db): {counts.size}')
print(f"Muestras por sujeto: {counts.min()} (mín) / {counts.max()} (máx)")
wnd_per = df.groupby(['group', 'stimulus']).size()
print(f"Muestras por sujeto x agarre: {wnd_per.min()} (mín) / {wnd_per.max()} (máx)")

Dataset:    ../data/prueba/intact_yj_mean_wnd.csv (DATASET = 'channels')
Estilo:     channel | 48 variables | unidad: ventanas
Meta:       ['origin_db', 'subject', 'stimulus', 'window']
Features:   ['Energy', 'Max', 'Peak_to_Peak', 'Range', 'VAR', 'VAR_STD']
Canales:    ['Channel 1', 'Channel 2', 'Channel 3', 'Channel 4', 'Channel 5', 'Channel 6', 'Channel 7', 'Channel 8']
Figuras en: ../figures/separabilidad_intact_yj_mean_wnd
Shape original: (16150, 52) | sin reposo: (16150, 52)
Clases: [22, 25, 27, 30, 31, 37, 40]


,n_muestras
stimulus,
22,2289
25,2318
27,2328
30,2318
31,2319
37,2290
40,2288


Grupos (subject_origin_db): 87
Muestras por sujeto: 175 (mín) / 211 (máx)
Muestras por sujeto x agarre: 25 (mín) / 31 (máx)


In [8]:
X = df[FEATURE_COLS].values.astype(float)
y = df['stimulus'].values
groups = df['group'].values

STIMULI = sorted(np.unique(y).tolist())
stim_color_map = dict(zip(STIMULI, STIM_COLORS))
CHANCE = 1 / len(STIMULI)
print(f'X: {X.shape} | clases: {STIMULI} | grupos: {len(np.unique(groups))} | azar = {CHANCE:.3f}')

X: (16150, 48) | clases: [22, 25, 27, 30, 31, 37, 40] | grupos: 87 | azar = 0.143


## 2. Fisher Discriminant Ratio (FDR)

FDR por variable: variabilidad **entre agarres** / variabilidad **dentro de cada agarre**.
Valor mayor → mejor separación. Se calcula también el **FDR global** (agregado sobre todas las
variables estandarizadas, para que todas pesen igual). La gráfica colorea cada barra por el tipo de feature extraído del nombre de la columna.


In [9]:
fdr_df, fdr_global_value = compute_fdr(X, y, FEATURE_COLS)
print(f'FDR global ({N_FEATURES} variables): {fdr_global_value:.4f}')
display(fdr_df.style.format({'FDR': '{:.4f}'}))

FDR global (48 variables): 0.1298


,Variable,FDR
1,Channel 2_Range,0.2262
2,Channel 8_Energy,0.1948
3,Channel 8_Max,0.1907
4,Channel 2_Max,0.1875
5,Channel 8_VAR_STD,0.1827
6,Channel 3_Max,0.1796
7,Channel 2_Energy,0.1771
8,Channel 6_Range,0.1758
9,Channel 3_Energy,0.1698
10,Channel 7_Max,0.1638


In [10]:
plot_fdr(fdr_df, fdr_global_value, info['feat_of'], info['feature_types'], SAVE_DIR)

## 3. LDA multiclase con LOSO

**Leave-One-Subject-Out:** en cada fold se deja fuera **un sujeto completo** para prueba y se
entrena con los 86 restantes (87 folds en total); así se mide si los agarres se separan **en
personas que el modelo no ha visto**.

- **Agrupamiento por `subject + origin_db`**: las filas de un sujeto quedan juntas. Si se
  barajaran por filas, el LDA se evaluaría con muestras del mismo sujeto que ya vio en
  entrenamiento y la accuracy sería optimista.
- **Cada sujeto se evalúa exactamente una vez** y el resultado es determinista (no depende de una
  semilla ni del número de repeticiones).
- `shrinkage='auto'` (Ledoit-Wolf): estabiliza la covarianza entre variables correlacionadas.
- Se reporta accuracy **global** y **por sujeto** (media ± desviación entre sujetos) frente al azar
  (1 / número de agarres).
- La matriz de confusión acumula las predicciones de prueba de **todos** los sujetos (cada fila del
  dataset aparece una vez); es la medida principal de **solapamiento entre agarres**.

La LDA aquí no es el clasificador final del proyecto: es un medidor lineal e interpretable de
separabilidad.


In [11]:
lda_res = run_lda_loso(X, y, groups)
cm = lda_res['cm']
per_group_df = lda_res['per_group_df']

print(f'LOSO ({len(per_group_df)} folds, un sujeto por fold) — LDA shrinkage={SHRINKAGE!r}')
print(f'Accuracy global (todas las predicciones de prueba): {lda_res["acc_global"]:.3f}')
print(f'Balanced accuracy:   {lda_res["bacc_global"]:.3f}')
print(f'Accuracy por sujeto: {per_group_df["accuracy"].mean():.3f} ± '
      f'{per_group_df["accuracy"].std():.3f} '
      f'(mín {per_group_df["accuracy"].min():.3f} / máx {per_group_df["accuracy"].max():.3f})')
print(f'Nivel de azar:       {CHANCE:.3f}')
print(f'Sujetos en o por debajo del azar: {(per_group_df["accuracy"] <= CHANCE).sum()} '
      f'de {len(per_group_df)}')

cm_df = pd.DataFrame(cm, index=STIMULI, columns=STIMULI)
cm_df.index.name = 'Real'
cm_df.columns.name = 'Predicho'

recall_df = pd.Series(np.diag(cm), index=STIMULI, name='Recall').to_frame()
recall_df.index.name = 'Estímulo'
display(recall_df.T.style.format('{:.3f}'))

LOSO (87 folds, un sujeto por fold) — LDA shrinkage='auto'
Accuracy global (todas las predicciones de prueba): 0.412
Balanced accuracy:   0.413
Accuracy por sujeto: 0.416 ± 0.157 (mín 0.109 / máx 0.766)
Nivel de azar:       0.143
Sujetos en o por debajo del azar: 2 de 87


Estímulo,22,25,27,30,31,37,40
Recall,0.409,0.336,0.334,0.265,0.599,0.452,0.493


In [12]:
plot_confusion(cm_df, per_group_df, CHANCE, SAVE_DIR)

# Confusiones mas frecuentes (fuera de la diagonal, simetrizadas)
conf_pairs = [{'agarre_a': STIMULI[a], 'agarre_b': STIMULI[b],
               'confusion': (cm[a, b] + cm[b, a]) / 2}
              for a, b in itertools.combinations(range(len(STIMULI)), 2)]
conf_pairs_df = (pd.DataFrame(conf_pairs).sort_values('confusion', ascending=False)
                 .reset_index(drop=True))
conf_pairs_df.index = conf_pairs_df.index + 1
print('Top-5 pares más confundidos en la LDA multiclase:')
display(conf_pairs_df.head(5).style.format({'confusion': '{:.3f}'}))

Top-5 pares más confundidos en la LDA multiclase:


,agarre_a,agarre_b,confusion
1,30,31,0.176
2,25,31,0.142
3,25,37,0.139
4,27,40,0.136
5,22,30,0.134


## 4. Matriz de solapamiento entre agarres (error LDA por pares)

Para cada par `(i, j)` se ajusta una LDA solo con esas dos clases y se mide el error con
**LOSO** (un fold por sujeto, igual que en la sección 3).

A diferencia de la matriz de confusión multiclase, esta mide cada par **aislado**, sin la
competencia de los demás agarres. La diagonal es 0. Valores altos ⇒ mayor solapamiento.


In [13]:
overlap = pairwise_overlap(X, y, STIMULI, groups)
n = len(STIMULI)
print(f'Error medio de los {n * (n - 1) // 2} pares: '
      f'{overlap[np.triu_indices(n, 1)].mean():.3f} (azar en un par = 0.5)')

Error medio de los 21 pares: 0.242 (azar en un par = 0.5)


In [14]:
overlap_df = pd.DataFrame(overlap, index=STIMULI, columns=STIMULI)
overlap_df.index.name = 'Estímulo'

plot_overlap(overlap_df, SAVE_DIR)

pairs = [{'agarre_a': STIMULI[a], 'agarre_b': STIMULI[b], 'error': overlap[a, b]}
         for a, b in itertools.combinations(range(n), 2)]
pairs_df = pd.DataFrame(pairs).sort_values('error', ascending=False).reset_index(drop=True)
pairs_df.index = pairs_df.index + 1
print('Top-5 pares más solapados (mayor error):')
display(pairs_df.head(5).style.format({'error': '{:.3f}'}))
print('Top-3 pares menos solapados:')
display(pairs_df.tail(3).style.format({'error': '{:.3f}'}))

Top-5 pares más solapados (mayor error):


,agarre_a,agarre_b,error
1,25,30,0.321
2,22,30,0.312
3,27,30,0.289
4,27,40,0.289
5,30,31,0.287


Top-3 pares menos solapados:


,agarre_a,agarre_b,error
19,27,31,0.185
20,31,37,0.152
21,31,40,0.145


## 5. Silhouette por clase

`silhouette_samples` con distancia euclídea sobre todas las variables **estandarizadas**, agrupado
por agarre.
Cerca de 1 → la muestra está más cerca de su agarre que de los demás; cerca de 0 o negativo →
solapamiento.

**Cuidado al interpretarlo:** si las filas son ventanas, las ~25 ventanas de un sujeto están
correlacionadas, así que la dispersión intra-clase se ve menor de lo que sería con una muestra por
sujeto. Además las features de amplitud (VAR, Energy, Range, Peak_to_Peak, Max, VAR_STD) están muy
correlacionadas y pesan más en la distancia euclídea. La LDA (sección 3) encuentra direcciones
discriminativas que la distancia euclídea no ve; por eso los dos resultados pueden discrepar.


In [15]:
sil_samples, sil_global, sil_class = compute_silhouette(X, y)
df['silhouette'] = sil_samples

print(f'Silhouette global: {sil_global:.4f}')
display(sil_class)

Silhouette global: -0.0615


,media,desv,n
Estímulo,,,
22,-0.0885,0.1091,2289
25,-0.1348,0.0830,2318
27,-0.1367,0.1136,2328
30,-0.0833,0.0807,2318
31,0.1143,0.1147,2319
37,-0.0561,0.1378,2290
40,-0.0452,0.1247,2288


In [16]:
plot_silhouette(sil_class, STIMULI, stim_color_map, sil_global, SAVE_DIR)

## 6. Proyección LDA (descriptiva)

LDA ajustada con **todos** los datos, solo para visualizar. Como usa todas las etiquetas, la
separación que se ve aquí es optimista: las cifras de separabilidad válidas son las de
LOSO (secciones 3 y 4). Máximo **min(n_clases − 1, n_variables)** componentes
discriminantes.

Dos figuras: 2D (LD1×LD2, `lda_projection.pdf`, con **centroides** por agarre) y 3D (LD1×LD2×LD3, `lda_projection_3d.pdf`, solo puntos).


In [17]:
X_lda, evr, var_df = lda_projection(X, y)

display(var_df.style.format({'Poder discriminante (λ)': '{:.4f}',
                             'Proporción': '{:.4f}', 'Acumulada': '{:.4f}'}))
print(f'LD1 + LD2 concentran el {evr[:2].sum() * 100:.2f}% del poder discriminante')
print(f'LD1 + LD2 + LD3 concentran el {evr[:3].sum() * 100:.2f}% del poder discriminante')

plot_lda_projection(X_lda, y, STIMULI, evr, STIM_COLORS, SAVE_DIR)
plot_lda_projection_3d(X_lda, y, STIMULI, evr, STIM_COLORS, SAVE_DIR)

,Componente,Poder discriminante (λ),Proporción,Acumulada
0,LD1,0.4409,0.3811,0.3811
1,LD2,0.2625,0.2269,0.6081
2,LD3,0.2379,0.2057,0.8137
3,LD4,0.1067,0.0922,0.9059
4,LD5,0.0665,0.0575,0.9634
5,LD6,0.0423,0.0366,1.0000


LD1 + LD2 concentran el 60.81% del poder discriminante
LD1 + LD2 + LD3 concentran el 81.37% del poder discriminante


## 7. Resumen automático

Se genera a partir de los resultados, para que no quede desactualizado si cambias `DATASET`,
o `SHRINKAGE`.


In [18]:
print_summary(info, fdr_df, fdr_global_value, lda_res, overlap,
              sil_global, sil_class, CHANCE)

Configuración: 48 variables (channel), SHRINKAGE='auto', LOSO, 87 sujetos, 7 agarres, 16150 ventanas

Fisher Discriminant Ratio
  FDR global = 0.130
  Variables más discriminativas: Channel 2_Range (0.226), Channel 8_Energy (0.195), Channel 8_Max (0.191)
  Variables menos discriminativas: Channel 4_VAR (0.042), Channel 4_VAR_STD (0.041)

LDA multiclase LOSO
  Accuracy global = 0.412 | por sujeto = 0.416 ± 0.157 (azar = 0.143); balanced accuracy = 0.413
  Mejor reconocido: 31 (0.599) | peor: 30 (0.265)
  Pares más confundidos: 30–31 (0.176), 25–31 (0.142), 25–37 (0.139)

Solapamiento por pares (LDA binaria LOSO)
  Error medio = 0.242
  Más solapados: 25–30 (0.321), 22–30 (0.312), 27–30 (0.289)
  Menos solapados: 27–31 (0.185), 31–37 (0.152), 31–40 (0.145)

Silhouette
  Global = -0.061; mejor clase 31 (0.114), peor 27 (-0.137)


In [19]:
sello.mostrar()

### Sello de procedencia

| Campo | Valor |
|---|---|
| Notebook | `07_separability_analysis.ipynb` |
| Commit | `07165f8` |
| Rama | `main` |
| Estado | sucio (3 archivos sin commitear) |
| Ejecutado | 2026-10-15T10:21:16 |

**Archivos generados en esta ejecución:**

- `figures/separabilidad_intact_yj_mean_wnd/fdr_features.pdf`
- `figures/separabilidad_intact_yj_mean_wnd/lda_loso_confusion.pdf`
- `figures/separabilidad_intact_yj_mean_wnd/lda_projection.pdf`
- `figures/separabilidad_intact_yj_mean_wnd/lda_projection_3d.pdf`
- `figures/separabilidad_intact_yj_mean_wnd/overlap_matrix.pdf`
- `figures/separabilidad_intact_yj_mean_wnd/silhouette_per_class.pdf`

> Guardado en `PROVENANCE.md` (raíz del repo).